In [2]:
import sys 
import os
from pathlib import Path 

import pandas as pd
import numpy as np

import calendar
from datetime import datetime
from datetime import date

import plotly.graph_objects as go


# Resolve path to project root
root_path = Path.cwd().parents[1]
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.queries.smhi import *

## Data query imports

- [ ] adjust how long format to wide format is transformed
- [ ] skip the imports and use data from global import

In [39]:
params_meteo={
    'version' : '1.0',
    'parameter' : ['1', '4', '6', '7', '39'],    # See here for more details https://opendata.smhi.se/metobs/resources/parameter
    'station' : '63510', # '63590'],    # This is the ID of the meteorolical station, which can be found here https://www.smhi.se/data/hitta-data-for-en-plats/ladda-ner-vaderobservationer/precipitationHourlySum
    'period' : 'corrected-archive',
    'data' : 'csv'
}

params_meteo_latest={
    'version' : '1.0',
    'parameter' : ['1', '4', '6', '7', '39'],    # See here for more details https://opendata.smhi.se/metobs/resources/parameter
    'station' : '63510', #'63590'],    # This is the ID of the meteorolical station, which can be found here https://www.smhi.se/data/hitta-data-for-en-plats/ladda-ner-vaderobservationer/precipitationHourlySum
    'period' : 'latest-months',
    'data'  : 'json'
}

df_metops = SMHI_Metops(params_meteo)
df_metops_latest = SMHI_Metops(params_meteo_latest)
df_metops = pd.merge(df_metops, df_metops_latest, how='outer')

Request SMHI Open Data Meteorological Observations.
Notice: 'corrected-archive' requested. Forcing data format to be CSV
Request SMHI Open Data Meteorological Observations.


In [49]:
params_hydro={
    'version' : 'latest',
    'parameter' : ['1', '3'],    # See here for more details https://opendata.smhi.se/hydroobs/resources/parameter
    'station' : ['1952'],     # This is the ID of the hydrological station, which can be found here https://www.smhi.se/data/hitta-data-for-en-plats/ladda-ner-observationer-fran-sjoar-och-vattendrag/waterLevel
    'period' : 'corrected-archive',
}

try:
    df_hydrops = SMHI_Hydroops(params_hydro)

    # Transform df_hydrops from long format into wide format
    df_hydrops_wide = df_hydrops.pivot_table(
        index='DateTime', 
        columns='StationName', 
        values=['Vattenföring (Dygn) [m³/s]'],
        aggfunc='mean'
    )

    # Flatten the multi-level df hydrops
    df_hydrops_wide.columns = [f"{metric}_{station}" for metric, station in df_hydrops_wide.columns]
    df_hydrops_wide = df_hydrops_wide.reset_index()


except ValueError:
    print(f"Hydrological Observations not loaded")

Request SMHI Open Data Hydrological Observations.
Hydrological Observations not loaded


## Plot creation

### Meteorology

#### Average Climate

In [50]:
df_tmp = df_metops.set_index('DateTime')

station_name = 'Ljunbgy A (63510)'

# Resample to Monthly sums/means
df_tmp = df_tmp.resample('MS').agg({
    'Lufttemperatur': 'mean',
    'Nederbördsmängd': 'sum'
})

# Create average data
df_tmp = df_tmp.groupby(df_tmp.index.month).mean()


fig = go.Figure()

fig.add_trace(
    go.Bar(
        x = df_tmp.index,
        y = df_tmp['Nederbördsmängd'],
        name='Precipitation', 
        yaxis='y'
    )
)
fig.add_trace(
    go.Scatter(
        x = df_tmp.index, 
        y = df_tmp['Lufttemperatur'], 
        name = 'Airtemperature',
        yaxis='y2',
    )
)

fig.update_layout(
    template='simple_white',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(
        #title='DateTime',
        tickvals = df_tmp.index,
        ticktext=calendar.month_name[1:]
    ),
    yaxis=dict(
        title='Precipitation [mm/month]',
        side='left', 
        range=[0, 100],
    ),
    yaxis2=dict(
        title='Airtemperature [°C]',
        side='right', 
        overlaying='y',
        range=[-5, 20]
    ), 
    legend=dict(
        xanchor='left', 
        x = 0.01, 
        yanchor='top', 
        y = 0.99
    )
)

# format the caption  
caption = f"Average climate at {station_name}.\
    Average annual precipitation: {df_tmp['Nederbördsmängd'].sum().round(2)} mm.\
        Average anual temperature: {df_tmp['Lufttemperatur'].mean().round(2)} °C."

## save caption
with open("../../fig/html/meteo_avg_bol_caption.md", "w") as f:
    f.write(caption)

fig.write_html("../../fig/html/meteo_avg_bol.html")

#### Current Climate

In [51]:
current_year = datetime.today().year

df_tmp = df_metops[df_metops['DateTime'].dt.year == current_year].set_index('DateTime')
station_name = 'Ljungby A (63510)'

month_names = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December",
}

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x = df_tmp.resample('ME').sum()['Nederbördsmängd'].index.month.map(month_names),
        y = df_tmp.resample('ME').sum()['Nederbördsmängd'],
        name = 'Precipitation',
        yaxis='y'
    )
)
fig.add_trace(
    go.Scatter(
        x = df_tmp.resample('ME').mean()['Lufttemperatur'].index.month.map(month_names),
        y = df_tmp.resample('ME').mean()['Lufttemperatur'],
        name = 'Airtemperature', 
        yaxis = 'y2'
    )
)

fig.update_layout(
    template='simple_white',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(
        #title='Month', 
    ),
    yaxis=dict(
        title='Precipitation [mm/month]', 
        side='left',
        range=[0, 100],
    ),
    yaxis2=dict(
        title='Airtemperature [°C]', 
        side='right', 
        overlaying='y',
        range=[-5, 20]
    ),
    legend=dict(
        xanchor='left', 
        x = 0.01, 
        yanchor='top', 
        y = 0.99
    )
)

# format the caption  
caption = f"Current climate at {station_name}.\
    Annual precipitation: {df_tmp['Nederbördsmängd'].sum().round(2)} mm.\
        Annual average temperature: {df_tmp['Lufttemperatur'].mean().round(2)} °C."

## save caption
with open("../../fig/html/meteo_current_bol_caption.md", "w") as f:
    f.write(caption)

fig.write_html("../../fig/html/meteo_current_bol.html")


### Limnology

#### Epilimnion

In [10]:
searchString = "####"
skip_until = None
with open(data_dir + 'SITES_CHE-L_BOL_BS-NW_20220510-20251128_L2_irregular.csv', "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        if searchString in line:
            skip_until = idx + 1
            break

if skip_until is None:
    raise ValueError(f"Marker {searchString} not found in file.")

df = pd.read_csv(
    data_dir + 'SITES_CHE-L_BOL_BS-NW_20220510-20251128_L2_irregular.csv',
    skiprows=skip_until
)

In [28]:
# get data 

data_dir = "../../data/"
searchString = "SITES_CHE-L_BOL_BS"

def read_SITES_csv(fname:str, headermarker:str="####", sitemarker:str="SAMPLING POINT:"):
    ''' Read SITES csv files and skips the header
    '''
    skip_until = None
    sampling_site = None
    with open(fname, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if sitemarker in line and sampling_site is None:
                sitename = line.split(sitemarker, 1)[1].rstrip("\r\n")

            if headermarker in line:
                skip_until = idx + 1
                break

    if skip_until is None:
        raise ValueError(f"Marker {headermarker} not found in file.")

    df = pd.read_csv(
        fname,
        skiprows=skip_until
    )
    df['SamplingSite'] = sitename

    return df

dfs = []
for fname in os.listdir(data_dir): 
    if searchString in fname:
        print(f"Reading: {fname}")
        
        tmp = read_SITES_csv(fname=data_dir + fname)
        dfs.append(tmp)

        del tmp

Reading: SITES_CHE-L_BOL_BS-NW_20220510-20251128_L2_irregular.csv
Reading: SITES_CHE-L_BOL_BS-E_20220510-20251128_L2_irregular.csv
Reading: SITES_CHE-L_BOL_BS-S_20221031-20251128_L2_irregular.csv
Reading: SITES_CHE-L_BOL_BS-NE_20220510-20251128_L2_irregular.csv
Reading: SITES_CHE-L_BOL_BS-W_20220510-20251128_L2_irregular.csv


In [30]:
df = pd.concat(dfs)

In [38]:
fig = go.Figure()

parameter = 'ABS420nm'
for site, group in df.groupby('SamplingSite', dropna=False):
    group = group.sort_values('TIMESTAMP')
    
    fig.add_trace(
        go.Scatter(
            x = group['TIMESTAMP'],
            y = group[parameter],
            mode = 'lines+markers', 
            name = str(site)
        )
    )

fig.update_layout(
    template='simple_white',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',

    xaxis_title="Timestamp",
    yaxis_title="ABS 420nm [1/m]",
    legend_title="Sampling Site",
)

# format the caption  
caption = f"Absorbance at 420 nm at all profiling stations"

## save caption
with open("../../fig/captions/CHE_Epilimnio_ABS420_bol.md", "w") as f:
    f.write(caption)

fig.write_html("../../fig/html/CHE_Epilimnio_ABS420_bol.html")